In [ ]:
from goto_sim.sim import Simulator, AltAzCache
from goto_sim.scheduling import HEATSurvey
from astropy.time import Time
import os

In [ ]:
sim = Simulator(start_time=Time("2026-01-01T00:00:00"), stop_time=Time("2027-01-01T00:00:00"))

In [ ]:
cache = sim.cache
cache_file = '2026_altaz_cache.npz'
# Check if cache file exists
if os.path.exists(cache_file):
    print(f"Loading cache from {cache_file}...")
    cache.load(cache_file)
else:
    print(f"Creating new cache and saving to {cache_file}...")
    cache.generate_cache()
    cache.save(cache_file)

In [ ]:
import numpy as np
from astropy import units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord, get_sun
from astropy.time import Time, TimeDelta
from time import perf_counter


def simulate_survey(
    start_date,
    end_date,
    tel_tile_masks,
    pointing_time=5 * 60,
    twilight=-12,
    min_alt=30,
):
    """Simulate a survey over the given time period."""
    start_time = perf_counter()
    print(f"Simulating from {start_date} to {end_date}...")

    # Create sites and grid
    sites = {
        "N": EarthLocation.of_site("lapalma"),
        "S": EarthLocation.of_site("sso"),
    }
    grid = SkyGrid.from_name("GOTO")

    # Run loop over time
    date = start_date
    observations = {t: [] for t in grid.tilenames}
    end_time = perf_counter()
    print(f"  Setup took {end_time - start_time:.2f} seconds.")
    while date < end_date:
        for site in sites:
            # First check if the Sun is above the horizon
            altaz_frame = AltAz(obstime=date, location=sites[site])
            sun = get_sun(date)
        date += TimeDelta(pointing_time * u.s)
    end_time = perf_counter()
    print(f"  Simulation took {end_time - start_time:.2f} seconds.")
    return observations


def simulate_survey_fast(
    start_date,
    end_date,
    tel_tile_masks,
    pointing_time=5 * 60,
    twilight=-12,
    min_alt=30,
):
    """Simulate a survey over the given time period."""
    start_time = perf_counter()
    print(f"Simulating from {start_date} to {end_date}...")

    # Create sites and grid
    sites = {
        "N": EarthLocation.of_site("lapalma"),
        "S": EarthLocation.of_site("sso"),
    }
    grid = SkyGrid.from_name("GOTO")

    # Run loop over time
    date = start_date
    observations = {t: [] for t in grid.tilenames}
    start_mjd = start_date.mjd
    end_mjd = end_date.mjd
    total_steps = int((end_mjd - start_mjd) * 86400 / pointing_time)
    times = Time(
        start_mjd + np.arange(total_steps) * pointing_time / 86400, format="mjd"
    )
    # Get sun position for all times
    sun_coords = get_sun(times)
    sun_altaz_dict = {}
    for site_key, site_location in sites.items():
        sun_altaz = sun_coords.transform_to(
            AltAz(obstime=times, location=site_location)
        )
        sun_altaz_dict[site_key] = sun_altaz
    lst_N = times.sidereal_time("apparent", sites["N"].lon)
    lst_S = times.sidereal_time("apparent", sites["S"].lon)
    end_time = perf_counter()
    print(f"  Setup took {end_time - start_time:.2f} seconds.")

    for date in times:
        for site in sites:
            if sun_altaz_dict[site][times == date].alt.deg > twilight:
                continue  # Skip if sun is above twilight limit

        pass  # Placeholder for the actual simulation logic
    end_time = perf_counter()
    print(f"  Simulation took {end_time - start_time:.2f} seconds.")
    return observations

In [ ]:
start_date = Time("2025-11-18 16:00")  # End of O4
end_date = Time("2025-12-18 16:00")

# Define which tiles each telescope can observe
heats_mask = np.array([tile in heats_tiles for tile in grid.tilenames])
cold_mask = np.array([tile in cold_tiles for tile in grid.tilenames])
tel_tile_masks = {
    1: heats_mask,
    2: cold_mask,
    3: heats_mask,
    4: cold_mask,
}

# Run the simulation
observations = simulate_survey(start_date, end_date, tel_tile_masks)

In [ ]:
# Run the simulation
observations = simulate_survey_fast(start_date, end_date, tel_tile_masks)

In [ ]:
start_mjd = start_date.mjd
end_mjd = end_date.mjd
pointing_time = 300
total_steps = int((end_mjd - start_mjd) * 86400 / pointing_time)
times = Time(start_mjd + np.arange(total_steps) * pointing_time / 86400, format="mjd")

In [ ]:
sidereal_times = times.sidereal_time("apparent", EarthLocation.of_site("lapalma"))
sidereal_times
# Round sidereal times to nearest second (angle)
sidereal_times_rounded = sidereal_times.round(3)  #
print(len(sidereal_times), len(set(sidereal_times_rounded)))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(sidereal_times)
plt.plot(sidereal_times_rounded, "o")

In [ ]:
sidereal_times.hour

In [ ]:
sidereal_times_rounded.hour

In [ ]:
# Find average time between sidereal time and rounded sidereal time
time_diffs = sidereal_times - sidereal_times_rounded
avg_diff = np.mean(np.abs(time_diffs))
print(avg_diff)

In [ ]:
# For each unique rounded sidereal time, find the position of each tile
from astropy.coordinates import Angle, HADec

unique_sidereal_times = Angle(sorted(set(sidereal_times_rounded)), unit=u.hourangle)

# We have SkyCoords for each tile from grid.coords, we can transform them to AltAz for each unique sidereal time
tile_altaz_dict = {}
coords = grid.coords


# Create a matrix of hour angles for each tile and each unique sidereal time
def compute_hour_angles(coords: SkyCoord, unique_sidereal_times: Angle) -> np.ndarray:
    hour_angles = np.zeros((len(unique_sidereal_times), len(coords))) * u.deg
    for i, lst in enumerate(unique_sidereal_times):
        hour_angles[i] = lst - coords.ra
    return hour_angles


hour_angles = compute_hour_angles(coords, unique_sidereal_times)
hour_angles

In [ ]:
hour_angles[0].shape

In [ ]:
hour_angles[0]

In [ ]:
hadec = HADec(ha=hour_angles, dec=coords.dec, location=EarthLocation.of_site("lapalma"))
hadec[750]

In [ ]:
# Get altitude from HADec
def hadec_to_altitude(hadec: HADec) -> Angle:
    altitude = np.atan2(np.sin(hadec.ha.rad))
    return altitude


altitudes = hadec_to_altitude(hadec)
altitudes

In [ ]:
# Compare altitudes with the values obtained from transforming to AltAz directly
idx = 0

altaz = coords.transform_to(
    AltAz(obstime=times[idx], location=EarthLocation.of_site("lapalma"))
)
# Remove negative altitudes for comparison
altaz = altaz[altaz.alt.deg >= 0]
altitudes = altitudes[idx][altitudes[idx].deg >= 0]

In [ ]:
diff = altitudes[idx] - altaz.alt
diff.deg

In [ ]:
plt.hist(diff.deg)

In [ ]:
altaz = coords[0].transform_to(
    AltAz(obstime=times, location=EarthLocation.of_site("lapalma"))
)

In [ ]:
# Create a grid of sidereal times across the whole angle range (0 to 24 hours) with 1 second resolution
sidereal_time_grid = Angle(np.arange(0, 24 * 3600) / 3600, unit=u.hourangle)
# Calculate hour angles for each tile and each sidereal time in the grid
hour_angles_grid = compute_hour_angles(coords, sidereal_time_grid)

In [ ]:
# Convert hour angles to altitudes
